# Build a Simple Agent

**Goal:** Create an AI agent that can reason, use tools, execute multi-step tasks, and pause for human approval. By the end, you'll understand the ReAct (Reasoning + Acting) pattern that powers real-world agent systems.

**API key optional** — With an OpenAI key, the agent uses GPT-4o-mini for reasoning. Without one, a rule-based fallback demonstrates the same architecture.

---

## What You'll Learn

1. How agents differ from chatbots (multi-step reasoning vs. single response)
2. The ReAct loop: Think → Act → Observe → Repeat
3. How to give an agent tools and let it decide which to use
4. Human-in-the-loop: when and how to pause for approval
5. Error handling and self-correction in agent workflows

In [ ]:
# Uncomment to install
# !pip install openai python-dotenv

import os
import json
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    from openai import OpenAI
    llm_client = OpenAI(api_key=api_key)
    HAS_LLM = True
    print("✓ LLM: OpenAI GPT-4o-mini (API)")
else:
    HAS_LLM = False
    print("⚠️ LLM: Not available (no API key). Rule-based agent will be used.")

print("\nSetup complete!")

## Step 1: Define Tools

An agent is only as useful as its tools. We'll give our agent four tools that query the consulting firm's Q3 data — the same data from previous notebooks.

In [ ]:
# ---- Tool implementations ----

def get_revenue_data(quarter, practice_area="all"):
    """Get revenue data for a given quarter and practice area."""
    data = {
        "Q3": {
            "total": 42.3, "yoy_growth": 12.0, "qoq_growth": -6.0,
            "by_practice": {
                "Defense": {"revenue": 15.2, "share": 36, "yoy": 18},
                "Financial Services": {"revenue": 12.8, "share": 30, "yoy": 5},
                "Health": {"revenue": 8.1, "share": 19, "yoy": 15},
                "Energy": {"revenue": 6.2, "share": 15, "yoy": 3},
            },
            "margin": 38.5, "backlog": 127.0, "bookings": 38.0,
            "book_to_bill": 0.90,
        },
        "Q2": {
            "total": 45.1, "yoy_growth": 15.0, "qoq_growth": 4.0,
            "by_practice": {
                "Defense": {"revenue": 14.0, "share": 31, "yoy": 14},
                "Financial Services": {"revenue": 14.5, "share": 32, "yoy": 12},
                "Health": {"revenue": 8.8, "share": 20, "yoy": 18},
                "Energy": {"revenue": 7.8, "share": 17, "yoy": 8},
            },
            "margin": 36.2, "backlog": 135.0, "bookings": 47.0,
            "book_to_bill": 1.04,
        },
    }
    if quarter not in data:
        return {"error": f"No data available for {quarter}. Available: Q2, Q3"}
    if practice_area != "all" and practice_area not in data[quarter]["by_practice"]:
        return {"error": f"Unknown practice area: {practice_area}"}
    if practice_area == "all":
        return data[quarter]
    return data[quarter]["by_practice"][practice_area]


def get_headcount_data(quarter):
    """Get headcount and talent data for a given quarter."""
    data = {
        "Q3": {"total": 852, "new_hires": 45, "departures": 17,
               "attrition_rate": 8.0, "utilization": 78, "engagement": 4.1,
               "offshore_team": 45},
        "Q2": {"total": 824, "new_hires": 38, "departures": 22,
               "attrition_rate": 10.0, "utilization": 82, "engagement": 4.1,
               "offshore_team": 38},
    }
    if quarter not in data:
        return {"error": f"No data available for {quarter}. Available: Q2, Q3"}
    return data[quarter]


def get_client_metrics(quarter):
    """Get client satisfaction and business development metrics."""
    data = {
        "Q3": {"active_clients": 94, "new_clients": 12, "retention": 92,
               "satisfaction": 4.7, "nps": 62, "avg_deal_size": 350,
               "pipeline_value": 89, "expansion_revenue": 14.2},
        "Q2": {"active_clients": 86, "new_clients": 8, "retention": 91,
               "satisfaction": 4.5, "nps": 58, "avg_deal_size": 563,
               "pipeline_value": 72, "expansion_revenue": 11.0},
    }
    if quarter not in data:
        return {"error": f"No data available for {quarter}. Available: Q2, Q3"}
    return data[quarter]


def calculate_growth(current, previous):
    """Calculate growth rate between two values."""
    if previous == 0:
        return {"error": "Cannot calculate growth from zero base"}
    rate = ((current - previous) / previous) * 100
    return {"current": current, "previous": previous, "growth_rate": round(rate, 1),
            "direction": "increase" if rate > 0 else "decrease"}


# Tool registry
TOOLS = {
    "get_revenue_data": {
        "function": get_revenue_data,
        "description": "Get revenue data for a quarter (Q2 or Q3). Can filter by practice area.",
        "parameters": {"quarter": "string (Q2 or Q3)", "practice_area": "string (optional, default 'all')"}
    },
    "get_headcount_data": {
        "function": get_headcount_data,
        "description": "Get headcount, hiring, attrition, and utilization data.",
        "parameters": {"quarter": "string (Q2 or Q3)"}
    },
    "get_client_metrics": {
        "function": get_client_metrics,
        "description": "Get client satisfaction, retention, NPS, and pipeline data.",
        "parameters": {"quarter": "string (Q2 or Q3)"}
    },
    "calculate_growth": {
        "function": calculate_growth,
        "description": "Calculate the percentage growth between two numeric values.",
        "parameters": {"current": "number", "previous": "number"}
    },
}

print(f"✓ Registered {len(TOOLS)} tools:")
for name, tool in TOOLS.items():
    print(f"   🔧 {name}: {tool['description']}")

## Step 2: The Agent Loop (ReAct Pattern)

The core of any agent is the **ReAct loop**: Reasoning + Acting.

```
User question
     │
     ▼
┌──────────┐
│  THINK   │ ← LLM decides what to do next
└────┬─────┘
     │
     ▼
┌──────────┐
│   ACT    │ ← Execute the chosen tool
└────┬─────┘
     │
     ▼
┌──────────┐
│ OBSERVE  │ ← Feed tool result back to LLM
└────┬─────┘
     │
     ├──► Need more info? → Loop back to THINK
     │
     └──► Have enough? → Generate final answer
```

In [ ]:
class SimpleAgent:
    """A ReAct agent that reasons about questions and uses tools to find answers."""
    
    def __init__(self, tools, require_approval_for=None):
        self.tools = tools
        self.require_approval_for = require_approval_for or []
        self.trace = []  # Execution trace for visualization
        
        self.system_prompt = f"""You are a senior analyst agent for a consulting firm's C-suite dashboard.
You have access to these tools:

{self._format_tools()}

To use a tool, respond with EXACTLY this format:
TOOL: tool_name
PARAMS: {{"param1": "value1", "param2": "value2"}}

To give a final answer, respond with:
ANSWER: your final answer here

Think step by step. Use tools to get data before answering. Never guess numbers."""
    
    def _format_tools(self):
        lines = []
        for name, tool in self.tools.items():
            params = json.dumps(tool['parameters'])
            lines.append(f"- {name}: {tool['description']} Params: {params}")
        return "\n".join(lines)
    
    def _parse_response(self, text):
        """Parse the LLM's response into an action."""
        text = text.strip()
        if "ANSWER:" in text:
            answer = text.split("ANSWER:", 1)[1].strip()
            return {"type": "answer", "content": answer}
        if "TOOL:" in text and "PARAMS:" in text:
            tool_line = text.split("TOOL:", 1)[1].split("\n")[0].strip()
            params_text = text.split("PARAMS:", 1)[1].strip()
            # Handle potential extra text after the JSON
            try:
                params = json.loads(params_text.split("\n")[0])
            except json.JSONDecodeError:
                params = {}
            return {"type": "tool_call", "tool": tool_line, "params": params}
        return {"type": "answer", "content": text}
    
    def _execute_tool(self, tool_name, params):
        """Execute a tool and return the result."""
        if tool_name not in self.tools:
            return {"error": f"Unknown tool: {tool_name}"}
        
        # Human-in-the-loop check
        if tool_name in self.require_approval_for:
            print(f"\n⏸️  HUMAN APPROVAL REQUIRED")
            print(f"   Agent wants to call: {tool_name}({params})")
            print(f"   [Simulating approval: APPROVED]")
            self.trace.append({"step": "approval", "tool": tool_name, "decision": "approved"})
        
        func = self.tools[tool_name]["function"]
        result = func(**params)
        return result
    
    def run(self, question, max_steps=5):
        """Run the agent loop on a question."""
        self.trace = []
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": question},
        ]
        
        print(f"\n{'='*60}")
        print(f"🤖 Agent received: \"{question}\"")
        print(f"{'='*60}")
        
        for step in range(max_steps):
            # THINK: Ask the LLM what to do
            if HAS_LLM:
                response = llm_client.chat.completions.create(
                    model="gpt-4o-mini", messages=messages,
                    temperature=0.1, max_tokens=500,
                )
                llm_text = response.choices[0].message.content
            else:
                llm_text = self._rule_based_think(question, step, messages)
            
            action = self._parse_response(llm_text)
            
            if action["type"] == "answer":
                print(f"\n💬 Final Answer (step {step + 1}):")
                print(f"   {action['content']}")
                self.trace.append({"step": "answer", "content": action["content"]})
                return action["content"]
            
            if action["type"] == "tool_call":
                tool_name = action["tool"]
                params = action["params"]
                
                print(f"\n🔧 Step {step + 1}: Calling {tool_name}({json.dumps(params)})")
                
                # ACT: Execute the tool
                result = self._execute_tool(tool_name, params)
                
                print(f"   📊 Result: {json.dumps(result)[:150]}")
                
                self.trace.append({"step": "tool_call", "tool": tool_name,
                                   "params": params, "result": result})
                
                # OBSERVE: Feed result back to LLM
                messages.append({"role": "assistant", "content": llm_text})
                messages.append({"role": "user",
                                 "content": f"Tool result: {json.dumps(result)}"})
        
        print(f"\n⚠️ Agent reached max steps ({max_steps}) without a final answer.")
        return None
    
    def _rule_based_think(self, question, step, messages):
        """Fallback for when no LLM is available — rule-based tool selection."""
        q = question.lower()
        if step == 0:
            if any(w in q for w in ['revenue', 'financial', 'margin', 'bookings', 'book-to-bill']):
                return 'TOOL: get_revenue_data\nPARAMS: {"quarter": "Q3"}'
            elif any(w in q for w in ['headcount', 'hiring', 'attrition', 'utilization', 'employee']):
                return 'TOOL: get_headcount_data\nPARAMS: {"quarter": "Q3"}'
            elif any(w in q for w in ['client', 'satisfaction', 'nps', 'pipeline', 'retention']):
                return 'TOOL: get_client_metrics\nPARAMS: {"quarter": "Q3"}'
            else:
                return 'TOOL: get_revenue_data\nPARAMS: {"quarter": "Q3"}'
        elif step == 1 and 'compare' in q or 'growth' in q or 'q2' in q:
            # Get comparison data
            last_result = messages[-1]['content'] if messages else ''
            if 'revenue' in q.lower():
                return 'TOOL: get_revenue_data\nPARAMS: {"quarter": "Q2"}'
            elif 'headcount' in q.lower():
                return 'TOOL: get_headcount_data\nPARAMS: {"quarter": "Q2"}'
            else:
                return 'TOOL: get_revenue_data\nPARAMS: {"quarter": "Q2"}'
        else:
            # Synthesize from what we have
            tool_results = [t['result'] for t in self.trace if t['step'] == 'tool_call']
            return f'ANSWER: Based on the data collected: {json.dumps(tool_results[0]) if tool_results else "No data available"}'


# Create the agent
agent = SimpleAgent(TOOLS)
print("✓ Agent created with ReAct loop and 4 tools")

## Step 3: Simple Task — Single Tool Call

Start simple: a question that requires just one tool call.

In [ ]:
agent.run("What was Q3 revenue?")

## Step 4: Multi-Step Reasoning

Now a harder question that requires multiple tool calls and computation — the agent must pull Q3 and Q2 data, then calculate the growth rate.

In [ ]:
agent.run("Compare Q3 revenue to Q2 and calculate the growth rate.")

In [ ]:
# Cross-domain question: needs data from multiple tools
agent.run("What's our client satisfaction and how does headcount relate to it?")

## Step 5: Human-in-the-Loop

In enterprise environments, certain actions should require human approval. Let's create an agent that pauses before executing specific tools.

In [ ]:
# Create agent that requires approval for revenue data access
supervised_agent = SimpleAgent(
    TOOLS,
    require_approval_for=["get_revenue_data"]  # Simulates approval for financial data
)

print("✓ Supervised agent created")
print("  Requires human approval for: get_revenue_data")
print("  (In production, this would prompt the user for confirmation)")
print()

supervised_agent.run("What was Q3 revenue by practice area?")

## Step 6: Error Handling and Self-Correction

What happens when a tool returns an error? A good agent detects the failure and tries a different approach.

In [ ]:
# Ask about a quarter that doesn't exist
agent.run("What was Q5 revenue?")

## Step 7: Visualize the Agent Trace

Every good agent system needs observability. Let's display the full execution trace — what the agent thought, what tools it called, and what it learned at each step.

In [ ]:
def visualize_trace(agent):
    """Display the agent's execution trace as an ASCII timeline."""
    print("\n" + "=" * 60)
    print("               AGENT EXECUTION TRACE")
    print("=" * 60)
    
    for i, step in enumerate(agent.trace):
        connector = "│" if i < len(agent.trace) - 1 else " "
        
        if step["step"] == "tool_call":
            print(f"  ┌─ 🔧 Tool Call: {step['tool']}")
            print(f"  │    Params: {json.dumps(step['params'])}")
            result_str = json.dumps(step['result'])[:80]
            print(f"  │    Result: {result_str}...")
            if 'error' in step.get('result', {}):
                print(f"  │    ❌ ERROR detected — agent will adapt")
            else:
                print(f"  │    ✓ Success")
            print(f"  {connector}")
        
        elif step["step"] == "approval":
            print(f"  ┌─ ⏸️  Human Approval: {step['tool']}")
            print(f"  │    Decision: {step['decision'].upper()}")
            print(f"  {connector}")
        
        elif step["step"] == "answer":
            print(f"  └─ 💬 Final Answer:")
            # Word wrap the answer
            answer = step['content']
            for line in [answer[i:i+55] for i in range(0, len(answer), 55)]:
                print(f"       {line}")
    
    print(f"\n  Total steps: {len(agent.trace)}")
    tool_calls = sum(1 for s in agent.trace if s['step'] == 'tool_call')
    print(f"  Tool calls: {tool_calls}")
    approvals = sum(1 for s in agent.trace if s['step'] == 'approval')
    if approvals:
        print(f"  Human approvals: {approvals}")


# Run a multi-step query and then visualize
agent.run("Compare Q3 and Q2 client metrics and calculate NPS growth.")
visualize_trace(agent)

## Step 8: Agent vs. Chatbot — The Difference

Let's make the distinction concrete by comparing what a chatbot does vs. what our agent does for the same question.

In [ ]:
question = "How does Q3 compare to Q2 across revenue, headcount, and client satisfaction?"

print("CHATBOT approach (single LLM call, no tools):")
print("-" * 50)
if HAS_LLM:
    chatbot_response = llm_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        temperature=0.1, max_tokens=300,
    )
    print(f"  {chatbot_response.choices[0].message.content[:300]}")
else:
    print("  [Without tools, the LLM would either hallucinate numbers")
    print("   or say 'I don't have access to your company data.']")

print(f"\n\nAGENT approach (ReAct loop with tools):")
print("-" * 50)
agent.run(question)

print(f"\n\n{'='*60}")
print("🎯 Key Difference:")
print("   Chatbot: Guesses or refuses (no data access)")
print("   Agent:   Retrieves real data, reasons across sources,")
print("           and provides a grounded answer with actual numbers.")

## Key Takeaways

1. **Agents execute workflows, chatbots answer questions.** The ReAct loop (Think → Act → Observe → Repeat) is what transforms an LLM from a responder into a doer.

2. **Tools make agents useful.** Without tools, an LLM can only work with its training data. With tools, it can query live databases, call APIs, and perform calculations on real data.

3. **Human-in-the-loop is essential for enterprise.** Certain actions (accessing financial data, sending communications, modifying records) should always require human approval. Build this into the agent architecture, not as an afterthought.

4. **Error handling is what separates demos from production.** When a tool returns an error, the agent should detect it, explain what went wrong, and either retry or adapt. Never silently fail.

5. **Observability is non-negotiable.** The execution trace (what the agent thought, which tools it called, what results it got) is critical for debugging, auditing, and building trust with users.

| Dimension | Chatbot | This Agent |
|---|---|---|
| **Data access** | Training data only | Live tools (4 registered) |
| **Reasoning** | Single response | Multi-step (ReAct loop) |
| **Error handling** | None | Detects and adapts |
| **Human oversight** | None | Approval gates |
| **Observability** | Response only | Full execution trace |

---

### Next Steps
- **[Agentic AI Fundamentals](../docs/07-agentic-ai.md)** — The conceptual foundation for agents
- **[Agent Orchestration Patterns](../docs/09-orchestration-patterns.md)** — Multi-agent architectures
- **[MCP (Model Context Protocol)](../docs/08-mcp.md)** — How agents connect to enterprise tools in production